In [1]:
import feedparser
from TextExtractor import TextExtractor

feed_url = "https://www.usu.edu/today/rss/UtahStateToday.xml"
feed = feedparser.parse(feed_url)

print(f"Feed Title: {feed.feed.title}\n")

def extract_interesting_rss_entry_fields(entry):
    article = {
        "title": entry.title,
        "category": entry.tags[0].term if len(entry.tags) > 0 else None
    }

    parser = TextExtractor()
    parser.feed(entry.summary)
    parser.close()

    article["summary"] = parser.text()

    return article

# display(feed.entries[0])

articles = list(map(extract_interesting_rss_entry_fields, feed.entries))
display(len(articles))

Feed Title: Utah State Today



50

In [3]:
# Extract all unique, non-null categories
unique_categories = {article["category"] for article in articles if article.get("category")}

# Print them sorted alphabetically
for category in sorted(unique_categories):
    print(category)

Arts &amp; Humanities
Athletics
Business & Society
Campus Life
Health & Wellness
Land & Environment
Science & Technology
Teaching & Learning
University Affairs


In [5]:
from sklearn.model_selection import train_test_split
# Splits the articles into 80% train and 20% test
train, test = train_test_split(articles, test_size=0.2, random_state=42)

print(f"Train size: {len(train)}")
print(f"Test size: {len(test)}")

Train size: 40
Test size: 10


In [6]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Extract the text strings from your list of dictionaries
train_texts = [article["summary"] for article in train]
test_texts = [article["summary"] for article in test]

# 2. Vectorize the text
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(train_texts)  # .fit_transform does fit and transform in one step
X_test_counts = count_vect.transform(test_texts)

# 3. View some vocabulary words
feature_names = count_vect.get_feature_names_out()
print("Sample vocabulary:", feature_names[:10])

# 4. See the shape of your training data matrix (Rows = articles, Columns = unique words)
print("X_train_counts shape:", X_train_counts.shape)

Sample vocabulary: ['00' '000' '01511' '026' '046' '0970' '10' '100' '1000' '1007']
X_train_counts shape: (40, 4193)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from sklearn import datasets
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB()

# Computing probabilities
classifier.fit(X_train_counts, [article["category"] for article in train])

# Use Bayes Theorem to predict
y_pred = classifier.predict(X_test_counts)
print(y_pred)
p,r,f,s = precision_recall_fscore_support([article["category"] for article in test], y_pred, average='weighted')
print(p,r,f)

['Land & Environment' 'Land & Environment' 'Science & Technology'
 'University Affairs' 'Science & Technology' 'Land & Environment'
 'Science & Technology' 'Business & Society' 'Land & Environment'
 'Science & Technology']
0.225 0.3 0.24


/home/ewalburger/CS_5830/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ewalburger/CS_5830/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
